# Post 009 — Time Series Forecasting
## Dataset A: Datacenter Power Demand Forecasting

**AI Engineering Lab Series | Era 1: Classic Machine Learning**

---

Datacenters buy electricity on spot markets. The price fluctuates hourly — sometimes by 10x between peak and off-peak hours. A datacenter that can accurately predict its own power demand 24 hours ahead can shift non-critical workloads (batch jobs, model training, backups) to cheap off-peak windows, saving millions annually.

This notebook demonstrates **ARIMA**, **SARIMA**, and **Facebook Prophet** for weekly power demand forecasting, covering stationarity testing, seasonal decomposition, and model selection.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
from statsmodels.tsa.stattools import adfuller, kpss
from statsmodels.tsa.seasonal import seasonal_decompose
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.tsa.statespace.sarimax import SARIMAX
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from sklearn.metrics import mean_absolute_error, mean_squared_error
import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-whitegrid')
print('Libraries loaded')

In [ ]:
df = pd.read_csv('../data/datacenter_power_demand.csv', parse_dates=['date'], index_col='date')
print(f'Shape: {df.shape}')
print(f'Date range: {df.index.min()} to {df.index.max()}')
print(f'\nBasic stats:')
print(df.describe())
df.head(10)

## 1. Visualize the Time Series

The first step in any time series analysis is to plot the raw data and look for:
- **Trend**: Is the series drifting up or down over time?
- **Seasonality**: Are there repeating patterns (weekly, monthly, annual)?
- **Noise**: How much random variation is there around the trend/seasonal pattern?

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(16, 8))

# Full series
axes[0].plot(df.index, df['power_mw'], color='steelblue', linewidth=1)
axes[0].set_title('Datacenter Power Demand: Full 3-Year History (Weekly)')
axes[0].set_ylabel('Power (MW)')
axes[0].xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))

# Last 52 weeks zoomed
last_year = df.iloc[-52:]
axes[1].plot(last_year.index, last_year['power_mw'], color='coral', linewidth=1.5)
axes[1].set_title('Last 52 Weeks: Weekly Seasonality Visible')
axes[1].set_ylabel('Power (MW)')
axes[1].xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))

plt.tight_layout()
plt.show()

## 2. Seasonal Decomposition

Seasonal decomposition splits the time series into three components:
- **Trend**: The long-term direction
- **Seasonal**: The repeating pattern
- **Residual**: What's left after removing trend and seasonality (ideally white noise)

This decomposition tells us which components ARIMA needs to model.

In [ ]:
decomposition = seasonal_decompose(df['power_mw'], model='additive', period=52)

fig, axes = plt.subplots(4, 1, figsize=(16, 12))
components = [('Observed', df['power_mw']), ('Trend', decomposition.trend),
              ('Seasonal (Annual)', decomposition.seasonal), ('Residual', decomposition.resid)]
colors = ['steelblue', 'coral', 'green', 'purple']

for ax, (title, data), color in zip(axes, components, colors):
    ax.plot(data.index, data.values, color=color, linewidth=1)
    ax.set_title(title)
    ax.set_ylabel('MW')
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))

plt.suptitle('Seasonal Decomposition: Datacenter Power Demand', fontsize=13)
plt.tight_layout()
plt.show()

## 3. Stationarity Testing

ARIMA requires the series to be **stationary** (constant mean and variance over time). We test this with the **Augmented Dickey-Fuller (ADF)** test. If the series is non-stationary, we difference it until it becomes stationary — that's the 'd' parameter in ARIMA(p, d, q).

In [ ]:
def adf_test(series, name='Series'):
    result = adfuller(series.dropna())
    print(f'{name}:')
    print(f'  ADF Statistic: {result[0]:.4f}')
    print(f'  p-value: {result[1]:.4f}')
    print(f'  Stationary: {"YES" if result[1] < 0.05 else "NO (needs differencing)"}')
    print()

adf_test(df['power_mw'], 'Original Series')
adf_test(df['power_mw'].diff(), 'First Difference')
adf_test(df['power_mw'].diff().diff(52), 'First + Seasonal Difference')

## 4. ACF and PACF: Choosing ARIMA Parameters

The **ACF (Autocorrelation Function)** and **PACF (Partial Autocorrelation Function)** plots tell us the p and q parameters for ARIMA:
- **PACF** cuts off sharply at lag p → use p for the AR order
- **ACF** cuts off sharply at lag q → use q for the MA order

In [ ]:
series_diff = df['power_mw'].diff().dropna()

fig, axes = plt.subplots(1, 2, figsize=(16, 4))
plot_acf(series_diff, lags=40, ax=axes[0], color='steelblue')
axes[0].set_title('ACF of Differenced Series (→ MA order q)')
plot_pacf(series_diff, lags=40, ax=axes[1], color='coral')
axes[1].set_title('PACF of Differenced Series (→ AR order p)')
plt.tight_layout()
plt.show()

## 5. ARIMA and SARIMA Forecasting

We split the last 26 weeks as a holdout test set and train on the remaining data. SARIMA adds seasonal terms P, D, Q with period m=52 (annual seasonality in weekly data).

In [ ]:
train = df['power_mw'].iloc[:-26]
test = df['power_mw'].iloc[-26:]
print(f'Train: {len(train)} weeks | Test: {len(test)} weeks')

# ARIMA(1,1,1)
arima_model = ARIMA(train, order=(1, 1, 1))
arima_fit = arima_model.fit()
arima_forecast = arima_fit.forecast(steps=26)

# SARIMA(1,1,1)(1,1,1,52)
sarima_model = SARIMAX(train, order=(1, 1, 1), seasonal_order=(1, 1, 1, 52))
sarima_fit = sarima_model.fit(disp=False)
sarima_forecast = sarima_fit.forecast(steps=26)

def eval_metrics(actual, predicted, name):
    mae = mean_absolute_error(actual, predicted)
    rmse = np.sqrt(mean_squared_error(actual, predicted))
    mape = np.mean(np.abs((actual - predicted) / actual)) * 100
    print(f'{name}: MAE={mae:.2f} MW | RMSE={rmse:.2f} MW | MAPE={mape:.2f}%')

eval_metrics(test.values, arima_forecast.values, 'ARIMA(1,1,1)      ')
eval_metrics(test.values, sarima_forecast.values, 'SARIMA(1,1,1)(1,1,1,52)')

In [ ]:
fig, ax = plt.subplots(figsize=(16, 6))

ax.plot(train.index[-52:], train.values[-52:], color='steelblue', label='Training (last year)', linewidth=1.5)
ax.plot(test.index, test.values, color='black', label='Actual', linewidth=2)
ax.plot(test.index, arima_forecast.values, color='coral', linestyle='--', label='ARIMA(1,1,1)', linewidth=1.5)
ax.plot(test.index, sarima_forecast.values, color='green', linestyle='--', label='SARIMA(1,1,1)(1,1,1,52)', linewidth=1.5)

ax.axvline(x=test.index[0], color='gray', linestyle=':', linewidth=1, label='Forecast start')
ax.set_title('Datacenter Power Demand: ARIMA vs SARIMA Forecast (26-Week Horizon)')
ax.set_ylabel('Power (MW)')
ax.legend()
ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))
plt.tight_layout()
plt.show()

## 6. Summary

| Model | MAE | RMSE | MAPE | Captures Seasonality? |
|---|---|---|---|---|
| **ARIMA(1,1,1)** | Higher | Higher | Higher | No |
| **SARIMA(1,1,1)(1,1,1,52)** | Lower | Lower | Lower | Yes |

**Key takeaways:**
1. **Stationarity first** — always test and difference before fitting ARIMA
2. **Seasonal data needs SARIMA** — ARIMA alone cannot capture annual patterns
3. **ACF/PACF are your parameter guides** — they tell you p and q directly
4. **MAPE is the most interpretable metric** — a 5% MAPE means forecasts are off by 5% on average, which translates directly to energy cost savings potential